# $T_\phi$ from second-order (curvature) flux noise

Combines the self-consistent analytic formulas (`Long/T2_second_order_flux_noise.ipynb`) with the
Toeplitz 2nd-order-cumulant integral (`paper plots/Second_order_flux_noise_T2.ipynb`) into a single
notebook that sweeps over multiple devices.

Coupling at the sweet spot: $\delta f_{01} = \tfrac12\kappa\,\delta\Phi^2$, $\kappa \equiv \partial^2 f_{01}/\partial\Phi^2$.
One-sided 1/f flux-noise PSD: $S_\Phi(f) = A_\Phi^2/f$ for $f\in[f_{\rm ir}, f_{\rm uv}]$. $f_{\rm uv}$ is
set to each device's qubit frequency $f_{01}$ (flux noise above $f_{01}$ drives $T_1$, not dephasing).

| Sequence | Formula | Method |
|---|---|---|
| Ramsey (exact, quasi-static) | $T_\phi^{\rm R} = \dfrac{\sqrt{e^4-1}}{2\pi|\kappa|A_\Phi^2\ln(f_{\rm max}/f_{\rm min})}$ | self-consistent |
| Echo, compact (Makhlin-Shnirman) | $\dfrac{1}{T_{2E}^{\rm compact}} = 2\pi|\kappa|A_\Phi^2\sqrt{\ln 2}$ | self-consistent |
| CPMG-$N$, compact | $\dfrac{1}{T_{\rm CPMG}} = \pi|\kappa|A_\Phi^2\sqrt{\ln 2\cdot\ln(f_{\rm max}T_{\rm CPMG}/N)}$ | self-consistent, swept over $N$ |
| Echo, Toeplitz | 2nd-order cumulant: $\mathrm{Var}(\phi)=2\lambda^2\Delta t^2\sum_d A(d)\,C(d\,\Delta t)^2$ | numerical integral |
| CPMG-1000, Toeplitz | same, with $N=1000$ refocusing pulses | numerical integral |

**Units:** $\kappa$ in Hz/$\mu\Phi_0^2$, $A_\Phi$ in $\mu\Phi_0/\sqrt{\rm Hz}$, $f$ in Hz.

**References:** Makhlin & Shnirman, PRL 92, 178301 (2004); Ithier et al., PRB 72, 134519 (2005);
Cywiński et al., PRB 77, 174509 (2008).


In [1]:
import numpy as np
import pandas as pd

_trapz = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz


## Device parameters — EDIT ME

Enter the curvature $\kappa$ (Hz/$\mu\Phi_0^2$) and qubit frequency $f_{01}$ (Hz) at the sweet spot
for each device. $f_{01}$ sets the UV cutoff of the dephasing integral (FLAG 4 in the source notebook).

In [2]:
devices = {
    'Device A': dict(kappa_Hz_uphi2=4, f01_Hz=16e6),   
    'Device B': dict(kappa_Hz_uphi2=192, f01_Hz=600e3),
    'Device C': dict(kappa_Hz_uphi2=432, f01_Hz=150e3),
}

# Noise model (shared across devices)
A_phi_uPhi0 = 1.0     # muPhi0 / sqrt(Hz), 1/f amplitude at 1 Hz
f_min_Hz    = 1.0     # Hz, IR cutoff ~ 1 / T_meas

for name, p in devices.items():
    print(f"{name}: kappa = {p['kappa_Hz_uphi2']} Hz/muPhi0^2,  f01 = {p['f01_Hz']/1e3:.1f} kHz")


Device A: kappa = 4 Hz/muPhi0^2,  f01 = 16000.0 kHz
Device B: kappa = 192 Hz/muPhi0^2,  f01 = 600.0 kHz
Device C: kappa = 432 Hz/muPhi0^2,  f01 = 150.0 kHz


## Self-consistent formulas — Ramsey, Echo (compact), CPMG-$N$ (compact)

Inputs to these functions use SI units: $\kappa$ in Hz/$\Phi_0^2$, $A_\Phi$ in $\Phi_0/\sqrt{\rm Hz}$.

In [3]:
def T_ramsey(kappa_Hz_phi2, A_phi_phi0, f_max, f_min):
    # Quasi-static Ramsey T_phi: exact chi-squared 1/e time.
    chi_pp = 2 * np.pi * kappa_Hz_phi2
    return np.sqrt(np.e**4 - 1) / (chi_pp * A_phi_phi0**2 * np.log(f_max / f_min))


def T_echo_compact(kappa_Hz_phi2, A_phi_phi0):
    # Compact Makhlin-Shnirman/Ithier echo estimate (no UV-log correction).
    chi_pp = 2 * np.pi * kappa_Hz_phi2
    rate   = chi_pp * A_phi_phi0**2 * np.sqrt(np.log(2))
    return 1.0 / rate


def T_cpmg_compact(kappa_Hz_phi2, A_phi_phi0, f_max, N=1, n_iter=40):
    # Self-consistent CPMG-N T2: 1/T = pi*|kappa|*A^2*sqrt(ln2 * ln(f_max*T/N)).
    chi_pp = 2 * np.pi * kappa_Hz_phi2
    C      = 0.5 * chi_pp * A_phi_phi0**2 * np.sqrt(np.log(2))   # = pi*kappa*A^2*sqrt(ln2)

    T = 1.0 / (chi_pp * A_phi_phi0**2 * np.sqrt(np.log(2)))      # seed: echo-compact value
    for _ in range(n_iter):
        arg = max(f_max * T / N, 1.0 + 1e-9)
        T   = 1.0 / (C * np.sqrt(np.log(arg)))
    return T


N_cpmg_compact_arr = np.linspace(1,1000,1000)


## Toeplitz 2nd-order-cumulant integral — Echo & CPMG-1000

Numerically integrates the refocused phase variance on a uniform time grid (Toeplitz structure),
without the rotating-wave / compact approximations above. Inputs use the $\mu\Phi_0$ convention
($\kappa$ in Hz/$\mu\Phi_0^2$, $A_\Phi$ in $\mu\Phi_0/\sqrt{\rm Hz}$).

In [4]:
N_cpmg_toeplitz = 1000

def toeplitz_solvers(f_ir, f_uv, A_phi_uPhi0, N_cpmg, t_grid=None,
                      pts_per_half_spacing=4, M_floor=4000):
    # Return (T_echo_fn, T_cpmg_fn): functions of kappa [Hz/muPhi0^2] -> T_phi [s].
    if t_grid is None:
        t_grid = np.logspace(-6, 3, 90)
    M = max(M_floor, 2 * pts_per_half_spacing * N_cpmg)

    f_grid = np.logspace(np.log10(f_ir), np.log10(f_uv), 400)
    lnf    = np.log(f_grid)

    def C_of_tau(tau):
        integrand = np.cos(2 * np.pi * np.outer(tau, f_grid))
        return A_phi_uPhi0**2 * _trapz(integrand, lnf, axis=1)

    def sign_sequence(kind, Mloc):
        frac = (np.arange(Mloc) + 0.5) / Mloc
        if kind == 'echo':
            n_before = (frac >= 0.5).astype(int)
        else:
            n_before = np.floor(frac * N_cpmg + 0.5).astype(int)
        return (-1.0) ** n_before

    def V0_of_t(kind, t):
        dt  = t / M
        s   = sign_sequence(kind, M)
        A_d = np.correlate(s, s, mode='full')[M - 1:]
        Cd2 = C_of_tau(np.arange(M) * dt) ** 2
        return 2.0 * np.pi**2 * dt**2 * (A_d[0] * Cd2[0] + 2.0 * np.sum(A_d[1:] * Cd2[1:]))

    V0tab = {k: np.array([V0_of_t(k, t) for t in t_grid]) for k in ('echo', 'cpmg')}

    def T_of(kind, kappa_Hz_uphi2):
        target = 2.0 / kappa_Hz_uphi2**2
        v = V0tab[kind]
        if v[-1] < target: return np.inf
        if v[0]  > target: return t_grid[0]
        j  = np.searchsorted(v, target)
        lt = np.log(t_grid[j-1:j+1]); lv = np.log(v[j-1:j+1])
        return float(np.exp(np.interp(np.log(target), lv, lt)))

    return (lambda kappa: T_of('echo', kappa)), (lambda kappa: T_of('cpmg', kappa))


def fmt(T):
    if not np.isfinite(T):
        return '   >range'
    for u, s in ((1e-3, 'us'), (1.0, 'ms'), (1e3, 's')):
        if T < u:
            return f'{T/(u/1e3):8.2f} {s}'
    return f'{T:8.2f} s'


## Compute $T_2$ for every device

In [5]:
results = []

for name, p in devices.items():
    kappa_uphi2 = p['kappa_Hz_uphi2']
    f01         = p['f01_Hz']

    kappa_phi2_SI = kappa_uphi2 * 1e12     # Hz/muPhi0^2 -> Hz/Phi0^2
    A_phi_phi0_SI = A_phi_uPhi0 * 1e-6     # muPhi0/sqrt(Hz) -> Phi0/sqrt(Hz)

    # -- self-consistent formulas (f_max = f01, f_min = f_min_Hz) --
    T_R  = T_ramsey(kappa_phi2_SI, A_phi_phi0_SI, f01, f_min_Hz)
    T_Ec = T_echo_compact(kappa_phi2_SI, A_phi_phi0_SI)

    T_cpmg_arr = np.array([
        T_cpmg_compact(kappa_phi2_SI, A_phi_phi0_SI, f01, N=N)
        for N in N_cpmg_compact_arr
    ])
    i_best   = int(np.argmax(T_cpmg_arr))
    T_Cc_max = T_cpmg_arr[i_best]
    N_best   = int(N_cpmg_compact_arr[i_best])

    # -- Toeplitz integral (f_uv = f01 per device) --
    T_echo_fn, T_cpmg_fn = toeplitz_solvers(f_min_Hz, f01, A_phi_uPhi0, N_cpmg_toeplitz)
    T_Et = T_echo_fn(kappa_uphi2)
    T_Ct = T_cpmg_fn(kappa_uphi2)

    results.append(dict(
        Device=name,
        kappa_Hz_uphi2=kappa_uphi2,
        f01_kHz=f01 / 1e3,
        T2_Ramsey_s=T_R,
        T2_EchoCompact_s=T_Ec,
        T2_CPMGCompact_s=T_Cc_max,
        N_CPMGCompact=N_best,
        T2_EchoToeplitz_s=T_Et,
        T2_CPMGToeplitz_s=T_Ct,
    ))

    print(f'{name}:  Ramsey={fmt(T_R)}  EchoCompact={fmt(T_Ec)}  '
          f'CPMGCompact(N={N_best})={fmt(T_Cc_max)}  '
          f'EchoToeplitz={fmt(T_Et)}  CPMGToeplitz(N={N_cpmg_toeplitz})={fmt(T_Ct)}')


Device A:  Ramsey=   17.56 ms  EchoCompact=   47.79 ms  CPMGCompact(N=1000)=   37.77 ms  EchoToeplitz=   41.84 ms  CPMGToeplitz(N=1000)=  450.69 ms
Device B:  Ramsey=  456.13 us  EchoCompact=  995.65 us  CPMGCompact(N=596)=    2.22 ms  EchoToeplitz=  533.78 us  CPMGToeplitz(N=1000)=   15.20 ms
Device C:  Ramsey=  226.30 us  EchoCompact=  442.51 us  CPMGCompact(N=66)=  985.54 us  EchoToeplitz=  223.74 us  CPMGToeplitz(N=1000)=    7.29 ms


Device B:  Ramsey=   95.46 ms  EchoCompact=  191.16 ms  CPMGCompact(N=1024)=  199.74 ms  EchoToeplitz=  334.73 ms  CPMGToeplitz(N=1000)=    2.23 s


Device C:  Ramsey=   95.46 ms  EchoCompact=  191.16 ms  CPMGCompact(N=1024)=  199.74 ms  EchoToeplitz=  334.73 ms  CPMGToeplitz(N=1000)=    2.23 s


## Results table

In [6]:
df = pd.DataFrame(results)

df_display = df.copy()
for col in ['T2_Ramsey_s', 'T2_EchoCompact_s', 'T2_CPMGCompact_s',
            'T2_EchoToeplitz_s', 'T2_CPMGToeplitz_s']:
    df_display[col.replace('_s', '')] = df_display[col].map(fmt)

df_display = df_display[['Device', 'kappa_Hz_uphi2', 'f01_kHz',
                          'T2_Ramsey', 'T2_EchoCompact', 'T2_CPMGCompact', 'N_CPMGCompact',
                          'T2_EchoToeplitz', 'T2_CPMGToeplitz']]
df_display.columns = ['Device', 'kappa (Hz/uPhi0^2)', 'f01 (kHz)',
                       'T2 Ramsey', 'T2 Echo (compact)', 'T2 CPMG (compact, best N)', 'N*',
                       'T2 Echo (Toeplitz)', f'T2 CPMG-{N_cpmg_toeplitz} (Toeplitz)']
df_display


,Device,kappa (Hz/uPhi0^2),f01 (kHz),T2 Ramsey,T2 Echo (compact),"T2 CPMG (compact, best N)",N*,T2 Echo (Toeplitz),T2 CPMG-1000 (Toeplitz)
0,Device A,4,16000.0,17.56 ms,47.79 ms,37.77 ms,1000,41.84 ms,450.69 ms
1,Device B,192,600.0,456.13 us,995.65 us,2.22 ms,596,533.78 us,15.20 ms
2,Device C,432,150.0,226.30 us,442.51 us,985.54 us,66,223.74 us,7.29 ms
